In [13]:
# DATA INGESTION FROM A WEBSITE

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.junglescout.com/resources/articles/introducing-speed-to-insight-complete-amazon-data-actionable-instantly/")
docs = loader.load() # docs is a list of documents

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [14]:
# splitting the docs

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 100)
chunks = splitter.split_documents(docs) # chunks is a list of documents

In [15]:
# create embedding model

from langchain_community.embeddings import OllamaEmbeddings
embedding = OllamaEmbeddings(model="nomic-embed-text")

/tmp/ipykernel_14476/2324702623.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding = OllamaEmbeddings(model="nomic-embed-text")


In [16]:
# create a FAISS vectr db and store the vectors of created chunks

from langchain_community.vectorstores import FAISS

vectorStoreDB=FAISS.from_documents(chunks,embedding)

In [17]:
# creating retriever for FAISS vectore store DB

retriever = vectorStoreDB.as_retriever(
    search_type="similarity",   # or "mmr"
    search_kwargs={"k": 3}      # number of chunks to retrieve
)

In [34]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
You are a helpful assistant.

Answer ONLY using the context.
If not found, say "I don't know".

Context:
{context}

Question:
{input}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "input"],
    template=prompt_template
)

### there are multiple models to pull from ollama in order to use them, for the time being i am using the low weight model means which is more quick to downlaod over the heavy models

In [28]:
# creating the llm
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model = "llama3.2:1b")


In [29]:
# creating the document chain and retreival chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

In [30]:
# creating the final RAG retrieval chain

from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [35]:
# Quering our RAG system

response = retrieval_chain.invoke({
    "input": "What is this website about?"
})

print(response["answer"])

This website appears to be about Jungle Scout, a tool used by brands and enterprises to improve their Amazon sales. It provides various insights and solutions for optimizing Amazon performance, including product strategy, market research, benchmarking, protection of market share, and launching new products. The website also shares success stories like the case study mentioned at the end.
